In [ ]:
# Imports
!pip install ucimlrepo
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import preprocessing, decomposition
import scipy
from scipy.sparse import hstack
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.model_selection import train_test_split

In [ ]:
# fetch dataset
bank_marketing = fetch_ucirepo(id=222)

# data (as pandas dataframes)
X = bank_marketing.data.features
y = bank_marketing.data.targets

# metadata
print(bank_marketing.metadata)

# variable information
print(bank_marketing.variables)

{'uci_id': 222, 'name': 'Bank Marketing', 'repository_url': 'https://archive.ics.uci.edu/dataset/222/bank+marketing', 'data_url': 'https://archive.ics.uci.edu/static/public/222/data.csv', 'abstract': 'The data is related with direct marketing campaigns (phone calls) of a Portuguese banking institution. The classification goal is to predict if the client will subscribe a term deposit (variable y).', 'area': 'Business', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 45211, 'num_features': 16, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Age', 'Occupation', 'Marital Status', 'Education Level'], 'target_col': ['y'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 2014, 'last_updated': 'Fri Aug 18 2023', 'dataset_doi': '10.24432/C5K306', 'creators': ['S. Moro', 'P. Rita', 'P. Cortez'], 'intro_paper': {'ID': 277, 'type': 'NATIVE', 'title': 'A data-driven approach to predict the s

# Handling missing values


In [ ]:
df = X.copy()
categorical_cols = ['job', 'marital', 'education', 'default', 'housing',
            'loan', 'contact', 'month', 'poutcome']
num_cols = ['age','balance','duration','campaign','pdays','previous']

# printing unknown rates
for i in categorical_cols:
  unknown_rate = (df[i].isna()).mean() * 100
  print(f"{i:10s}: {unknown_rate:.4f}%")

print("missing values in numeric before:".upper())
print(df[num_cols].isna().sum()) #no missing values in numerical columns so no need to fill in

for col in ['job', 'education']: #replace nan's with unknown as it should be according to the dataset description
    df[col] = df[col].fillna('unknown')

df['contact'] = df['contact'].fillna('none')
print("missing values in categorical after:".upper())
for i in categorical_cols:
  unknown_rate = (df[i].isna()).mean() * 100
  print(f"{i:10s}: {unknown_rate:.4f}%")

print("missing values in numeric after:".upper())
print(df[num_cols].isna().sum()) #no missing values in numerical columns so no need to fill in

df = df.drop(columns=['poutcome']) # drop poutcome since it is not important to teh dataset and missing a high number ov values



job       : 0.6370%
marital   : 0.0000%
education : 4.1074%
default   : 0.0000%
housing   : 0.0000%
loan      : 0.0000%
contact   : 28.7983%
month     : 0.0000%
poutcome  : 81.7478%
MISSING VALUES IN NUMERIC BEFORE:
age         0
balance     0
duration    0
campaign    0
pdays       0
previous    0
dtype: int64
MISSING VALUES IN CATEGORICAL AFTER:
job       : 0.0000%
marital   : 0.0000%
education : 0.0000%
default   : 0.0000%
housing   : 0.0000%
loan      : 0.0000%
contact   : 0.0000%
month     : 0.0000%
poutcome  : 81.7478%
MISSING VALUES IN NUMERIC AFTER:
age         0
balance     0
duration    0
campaign    0
pdays       0
previous    0
dtype: int64


# Convert Categorical attributes and target to numerical

In [ ]:
df = X.copy()
target = y.copy()

# one-hot encode categorical attributes (removed poutcome)
cat_atts = ['job',
            'marital',
            'education',
            'default',
            'housing',
            'loan',
            'contact'
            ]

y_enc = preprocessing.OrdinalEncoder()
y_vector = y_enc.fit_transform(target['y'].to_frame())

encs = dict()
cat_vectorized = dict()
for att in cat_atts:
  encs[att] = preprocessing.OneHotEncoder()
  cat_vectorized[att] = encs[att].fit_transform(df[att].to_frame())

# enumerate month attribute (jan = 0, feb = 1... dec = 11)
months = ['jan', 'feb', 'mar', 'apr', 'may', 'jun',
         'jul', 'aug', 'sep', 'oct', 'nov', 'dec']
month = df['month']
for num, name in enumerate(months):
  month = month.map(lambda x: num if x == name else x)


# Vectorization

In [ ]:
# no need to change numeric attributes
age = df['age'];
balance = df['balance']
day_of_week = df['day_of_week']
duration = df['duration']
campaign = df['campaign']
pdays = df['pdays']
previous = df['previous']

# vectorize all attributes                            # dimension of attribute
attributes = [scipy.sparse.csr_matrix(age.values).T,  #  1
              cat_vectorized['job'],                  # 12
              cat_vectorized['marital'],              #  3
              cat_vectorized['education'],            #  4
              cat_vectorized['default'],              #  2
              scipy.sparse.csr_matrix(balance).T,     #  1
              cat_vectorized['housing'],              #  2
              cat_vectorized['loan'],                 #  2
              cat_vectorized['contact'],              #  3
              scipy.sparse.csr_matrix(day_of_week).T, #  1
              scipy.sparse.csr_matrix(month).T,       #  1
              scipy.sparse.csr_matrix(duration).T,    #  1
              scipy.sparse.csr_matrix(campaign).T,    #  1
              scipy.sparse.csr_matrix(pdays).T,       #  1
              scipy.sparse.csr_matrix(previous).T,    #  1
              ]                                       # sum: 36

# initialize normalizer
normalizers = [None] * len(attributes)

# normalize all attributes
for i in range(len(attributes)):
  normalizers[i] = preprocessing.Normalizer().fit(attributes[i])
  attributes[i] = normalizers[i].transform(attributes[i])

# merge all attributes into one matrix
vectorized = hstack(attributes)
print(vectorized.shape)

print(y_vector.shape)
print(y_vector)


(45211, 36)
(45211, 1)
[[0.]
 [0.]
 [0.]
 ...
 [1.]
 [0.]
 [0.]]


# Splitting dataset into train and test




In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    vectorized,          # attributes
    y,                   # target
    test_size=0.2,       # 20% test 80% training
    random_state=40
)

print(type(X_train), type(X_test), type(y_train), type(y_test))
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

<class 'scipy.sparse._csr.csr_matrix'> <class 'scipy.sparse._csr.csr_matrix'> <class 'pandas.core.frame.DataFrame'> <class 'pandas.core.frame.DataFrame'>
(36168, 36) (9043, 36) (36168, 1) (9043, 1)


# Boosting to reduce imbalanced data

In [ ]:
#  Regular SMOTE
# from imblearn.over_sampling import SMOTE
# smote = SMOTE(random_state=42)
# X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

#  Borderline-SMOTE
# from imblearn.over_sampling import BorderlineSMOTE
# bsmote = BorderlineSMOTE(random_state=42)
# X_train_bal, y_train_bal = bsmote.fit_resample(X_train, y_train)

# ADASYN
# from imblearn.over_sampling import ADASYN
# adasyn = ADASYN(random_state=42)
# X_train_bal, y_train_bal = adasyn.fit_resample(X_train, y_train)

# Random Undersampling
# from imblearn.under_sampling import RandomUnderSampler
# rus = RandomUnderSampler(random_state=42)
# X_train_bal, y_train_bal = rus.fit_resample(X_train, y_train)

# # SMOTE-Tomek
# from imblearn.combine import SMOTETomek
# smt = SMOTETomek(random_state=42)
# X_train_bal, y_train_bal = smt.fit_resample(X_train, y_train)

# # SMOTE-ENN
from imblearn.combine import SMOTEENN
sme = SMOTEENN(random_state=42)
X_train_bal, y_train_bal = sme.fit_resample(X_train, y_train)

# Projection Component Analysis and normalizing

In [ ]:
n = 15 # number of projected dimensions

pca = decomposition.PCA(n_components=n)
pca.fit(X_train)
pca.set_output(transform='pandas') # we want pandas df output

training_data = pca.transform(X_train_bal)
test_data = pca.transform(X_test)

print(type(training_data), type(test_data))
print(training_data.shape, test_data.shape)

<class 'pandas.core.frame.DataFrame'> <class 'pandas.core.frame.DataFrame'>
(26669, 15) (9043, 15)


# KNN

In [ ]:
import statistics

# KNN takes a matrix X of entries, each associated with a class in Y
#     and returns the (voted) average class of the k nearest neighbours
#     of x
#
#     For all inputs, expect numpy arrays
#     X[N][d]: 'training' example matrix
#     Y[N]: classes of X
#     x[d]: test example

def KNN(X, Y, x, k):
  diff = X-x # column wise subtraction of vector x from matrix X
  norms = np.sqrt(np.sum(diff * diff, axis=1))

  indices = norms.argsort()[:k] # indices of k least norms

  classes = Y[indices].ravel() # converts [n,1] to [n]

  return statistics.mode(classes)

In [ ]:
def test_KNN(X, y, X_test, Y_test, k):
  num = X_test.shape[0]
  fn = 0
  fp = 0
  tn = 0
  tp = 0

  for rowNum in range(num):
    row = np.reshape(X_test[rowNum], (1,n))
    predicted = KNN(X, y, row, k)
    actual = Y_test[rowNum]
    if predicted == actual:
      if actual == 'yes':
        tp += 1
      else: # actual == 'no'
        tn += 1
    else:
      if predicted == 'yes':
        fp += 1
      else:
        fn += 1

  return tp, fp, fn, tn

# Prediction on Test Set (Done with best found Hyperparameters)

We chose to bias the classification towards the minority class ('yes'), resulting in better recall with the drawback of lower accuracy.

In [ ]:
num = test_data.shape[0]
k = 7w

trainData = training_data.to_numpy()
trainClasses = y_train_bal.to_numpy()
testData = test_data.to_numpy()
testClasses = y_test.to_numpy()

tp, fp, fn, tn = test_KNN(trainData, trainClasses, testData, testClasses, k)


print(f'tp: {tp/num * 100.0:.2f}%\nfp: {fp/num * 100.0:.2f}%\nfn: {fn/num * 100.0:.2f}%\ntn: {tn/num * 100.0:.2f}%')



tp: 7.57%
fp: 37.13%
fn: 4.22%
tn: 51.07%


In [ ]:
accuracy = (tp+tn)/(tp+tn+fp+fn)
precision = tp/(tp+fp)
recall = tp/(tp+fn)
fscore = (2*precision*recall)/(precision+recall)

print(f'accuracy:{accuracy}')
print(f'precision:{precision}')
print(f'recall:{recall}')
print(f'fscore:{fscore}')

accuracy:0.5864204356961186
precision:0.1694286420974524
recall:0.6419868791002812
fscore:0.26810176125244617


In [ ]:
# save training data for KNN
with open('KNN_data.npy', 'wb') as f:
    np.save(f, trainData)
    np.save(f, trainClasses)

# save small subset (100 examples)
subset_data = testData[123:223]
subset_classes = testClasses[123:223]

with open('test_subset.npy', 'wb') as f:
    np.save(f, subset_data)
    np.save(f, subset_classes)


# Fast Track Prediction

To bypass full running of the entire colab, load the following files into the colab directory:
- <code>KNN_data.npy</code>
- <code>test_subset.npy</code>

Then, run the first two cells of "**KNN**" (two functions.)

Finally, run the code cell below.

In [ ]:
import numpy as np
n=15
# load KNNData
with open('KNN_data.npy', 'rb') as f:
    X = np.load(f, allow_pickle=True)
    y = np.load(f, allow_pickle=True)

with open('test_subset.npy', 'rb') as f:
    X_fast_test = np.load(f, allow_pickle=True)
    y_fast_test = np.load(f, allow_pickle=True)

tp, fp, fn, tn = test_KNN(X, y, X_fast_test, y_fast_test, 7) # KNN with k=7

print(f'tp: {tp}')
print(f'fp: {fp}')
print(f'fn: {fn}')
print(f'tn: {tn}')
print('---')

accuracy = (tp+tn)/(tp+tn+fp+fn)
precision = tp/(tp+fp)
recall = tp/(tp+fn)
fscore = (2*precision*recall)/(precision+recall)

print(f'accuracy:{accuracy}')
print(f'precision:{precision}')
print(f'recall:{recall}')
print(f'fscore:{fscore}')


tp: 10
fp: 24
fn: 1
tn: 65
---
accuracy:0.75
precision:0.29411764705882354
recall:0.9090909090909091
fscore:0.4444444444444445


# Alternative Approaches

##Testing different balancing methods

In [ ]:
from imblearn.over_sampling import SMOTE, BorderlineSMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek, SMOTEENN
from sklearn import decomposition
import numpy as np
import pandas as pd

balancers = {
    "SMOTE": SMOTE(random_state=42),
    "BorderlineSMOTE": BorderlineSMOTE(random_state=42),
    "ADASYN": ADASYN(random_state=42),
    "RandomUnderSampler": RandomUnderSampler(random_state=42),
    "SMOTETomek": SMOTETomek(random_state=42),
    "SMOTEENN": SMOTEENN(random_state=42)
}

results = {}
n = 15             # PCA dimensions
k = 3                # number of neighbors

print(f" n={n} --------- k={k} \n")
for name, sampler in balancers.items():
    print(f"\n=== {name} ===")

    # Rebalance
    X_train_bal, y_train_bal = sampler.fit_resample(X_train, y_train)

    # PCA transform (fit only once on original X_train)
    pca = decomposition.PCA(n_components=n)
    pca.fit(X_train)
    pca.set_output(transform='pandas')

    training_data = pca.transform(X_train_bal)
    test_data = pca.transform(X_test)

    trainData = training_data.to_numpy()
    trainClasses = y_train_bal.to_numpy()
    testData = test_data.to_numpy()
    testClasses = y_test.to_numpy()

    # Confusion counts
    tp = fp = fn = tn = 0
    num = testData.shape[0]

    for rowNum in range(num):
        row = np.reshape(testData[rowNum], (1, n))
        predicted = KNN(trainData, trainClasses, row, k)
        actual = testClasses[rowNum]
        if predicted == actual:
            if actual == 'yes':
                tp += 1
            else:
                tn += 1
        else:
            if predicted == 'yes':
                fp += 1
            else:
                fn += 1

    # Normalized confusion and metrics
    total = tp + tn + fp + fn
    tp_n, fp_n, fn_n, tn_n = tp/total, fp/total, fn/total, tn/total
    acc = (tp + tn) / total
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1 = 2*prec*rec / (prec + rec) if (prec + rec) > 0 else 0

    results[name] = [tp_n, fp_n, fn_n, tn_n, acc, prec, rec, f1]

    print(f"tp: {tp_n:.4f}, fp: {fp_n:.4f}, fn: {fn_n:.4f}, tn: {tn_n:.4f}")
    print(f"Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}")

# Summary table
df_results = pd.DataFrame(results, index=["TP", "FP", "FN", "TN", "Accuracy", "Precision", "Recall", "F1"]).T
print(f"\n=== Overall Comparison of Balancing Methods  k={k}  n={n} ===")
print(df_results.round(4))


## No PCA

In [ ]:
from imblearn.over_sampling import SMOTE, BorderlineSMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek, SMOTEENN
import numpy as np
import pandas as pd
from scipy.sparse import issparse

def to_dense_array(x):
    if hasattr(x, "to_numpy"):
        arr = x.to_numpy()
    elif issparse(x):
        arr = x.toarray()
    else:
        arr = np.asarray(x)
    if arr.ndim == 1:
        arr = arr.reshape(-1, 1)
    return arr

# Define balancing methods
balancers = {
    "SMOTE": SMOTE(random_state=42),
    "BorderlineSMOTE": BorderlineSMOTE(random_state=42),
    "ADASYN": ADASYN(random_state=42),
    "RandomUnderSampler": RandomUnderSampler(random_state=42),
    "SMOTETomek": SMOTETomek(random_state=42),
    "SMOTEENN": SMOTEENN(random_state=42)
}

results = {}
k = 5  # number of neighbors

print(f"\nNo PCA --------- k={k}\n")

for name, sampler in balancers.items():
    print(f"\n=== {name} ===")

    # Rebalance the training data
    X_train_bal, y_train_bal = sampler.fit_resample(X_train, y_train)

    # Convert
    trainData = to_dense_array(X_train_bal)
    trainClasses = np.asarray(y_train_bal)
    testData = to_dense_array(X_test)
    testClasses = np.asarray(y_test)

    n = trainData.shape[1]  # number of features
    num = testData.shape[0]

    # Confusion counts
    tp = fp = fn = tn = 0

    for rowNum in range(num):
        row = np.reshape(testData[rowNum], (1, n))
        predicted = KNN(trainData, trainClasses, row, k)
        actual = testClasses[rowNum]

        if predicted == actual:
            if actual == 'yes':
                tp += 1
            else:
                tn += 1
        else:
            if predicted == 'yes':
                fp += 1
            else:
                fn += 1

    # Normalized confusion values and metrics
    total = tp + tn + fp + fn
    tp_n, fp_n, fn_n, tn_n = tp/total, fp/total, fn/total, tn/total
    acc = (tp + tn) / total
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0

    results[name] = [tp_n, fp_n, fn_n, tn_n, acc, prec, rec, f1]

    print(f"tp: {tp_n:.4f}, fp: {fp_n:.4f}, fn: {fn_n:.4f}, tn: {tn_n:.4f}")
    print(f"Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}")

# Summary comparison table
df_results = pd.DataFrame(results,
    index=["TP", "FP", "FN", "TN", "Accuracy", "Precision", "Recall", "F1"]
).T

print(f"\n=== Overall Comparison of Balancing Methods  k={k}  (No PCA) ===")
print(df_results.round(4))

## Summary of Various Parameters

```
=== Overall Comparison of Balancing Methods  k=5   n=20 ===
                        TP      FP      FN      TN  Accuracy  Precision   
SMOTE               0.0176  0.0535  0.1004  0.8285    0.8461     0.2473   
BorderlineSMOTE     0.0176  0.0412  0.1004  0.8408    0.8583     0.2989   
ADASYN              0.0201  0.0581  0.0979  0.8240    0.8441     0.2574   
RandomUnderSampler  0.0749  0.3536  0.0431  0.5284    0.6032     0.1747   
SMOTETomek          0.0212  0.0530  0.0968  0.8290    0.8503     0.2861   
SMOTEENN            0.0853  0.4507  0.0327  0.4313    0.5165     0.1591   

                    Recall      F1  
SMOTE               0.1490  0.1860  
BorderlineSMOTE     0.1490  0.1989  
ADASYN              0.1706  0.2052  
RandomUnderSampler  0.6345  0.2740  
SMOTETomek          0.1799  0.2209  
SMOTEENN            0.7226  0.2607  


=== Overall Comparison of Balancing Methods  k=5  n=30 ===
                        TP      FP      FN      TN  Accuracy  Precision  \
SMOTE               0.0258  0.0683  0.0922  0.8137    0.8394     0.2738   
BorderlineSMOTE     0.0164  0.0502  0.1016  0.8318    0.8482     0.2458   
ADASYN              0.0238  0.0813  0.0942  0.8007    0.8245     0.2263   
RandomUnderSampler  0.0738  0.3299  0.0442  0.5521    0.6259     0.1827   
SMOTETomek          0.0207  0.0604  0.0973  0.8216    0.8423     0.2551   
SMOTEENN            0.0853  0.4606  0.0327  0.4214    0.5067     0.1562   

                    Recall      F1  
SMOTE               0.2184  0.2430  
BorderlineSMOTE     0.1387  0.1774  
ADASYN              0.2015  0.2132  
RandomUnderSampler  0.6251  0.2828  
SMOTETomek          0.1753  0.2078  
SMOTEENN            0.7226  0.2569  


=== Overall Comparison of Balancing Methods  k=3  n=20 ===
                        TP      FP      FN      TN  Accuracy  Precision  \
SMOTE               0.0281  0.0874  0.0899  0.7946    0.8227     0.2433   
BorderlineSMOTE     0.0218  0.0458  0.0962  0.8362    0.8580     0.3224   
ADASYN              0.0230  0.0623  0.0950  0.8198    0.8428     0.2698   
RandomUnderSampler  0.0682  0.3436  0.0498  0.5384    0.6067     0.1657   
SMOTETomek          0.0229  0.0714  0.0951  0.8106    0.8335     0.2427   
SMOTEENN            0.0855  0.4535  0.0325  0.4285    0.5140     0.1586   

                    Recall      F1  
SMOTE               0.2381  0.2406  
BorderlineSMOTE     0.1846  0.2348  
ADASYN              0.1949  0.2263  
RandomUnderSampler  0.5783  0.2576  
SMOTETomek          0.1940  0.2156  
SMOTEENN            0.7245  0.2602


=== Overall Comparison of Balancing Methods  k=5  n=15 ===
                        TP      FP      FN      TN  Accuracy  Precision  \
SMOTE               0.0203  0.0484  0.0976  0.8336    0.8539     0.2958   
BorderlineSMOTE     0.0186  0.0487  0.0994  0.8334    0.8519     0.2763   
ADASYN              0.0217  0.0668  0.0963  0.8152    0.8369     0.2450   
RandomUnderSampler  0.0686  0.3013  0.0494  0.5807    0.6492     0.1854   
SMOTETomek          0.0218  0.0637  0.0962  0.8183    0.8401     0.2549   
SMOTEENN            0.0766  0.3721  0.0414  0.5099    0.5865     0.1708   

                    Recall      F1  
SMOTE               0.1724  0.2179  
BorderlineSMOTE     0.1575  0.2006  
ADASYN              0.1837  0.2100  
RandomUnderSampler  0.5811  0.2811  
SMOTETomek          0.1846  0.2141  
SMOTEENN            0.6495  0.2704  


=== Overall Comparison of Balancing Methods  k=7  n=15 ===
                        TP      FP      FN      TN  Accuracy  Precision  \
SMOTE               0.0171  0.0417  0.1009  0.8403    0.8575     0.2914   
BorderlineSMOTE     0.0198  0.0429  0.0982  0.8391    0.8589     0.3157   
ADASYN              0.0171  0.0418  0.1009  0.8402    0.8573     0.2908   
RandomUnderSampler  0.0748  0.3115  0.0432  0.5705    0.6453     0.1935   
SMOTETomek          0.0248  0.0579  0.0932  0.8241    0.8488     0.2995   
SMOTEENN            0.0757  0.3713  0.0422  0.5107    0.5864     0.1694   

                    Recall      F1  
SMOTE               0.1453  0.1939  
BorderlineSMOTE     0.1678  0.2191  
ADASYN              0.1453  0.1937  
RandomUnderSampler  0.6336  0.2965  
SMOTETomek          0.2099  0.2468  
SMOTEENN            0.6420  0.2681  


 n=10 --------- k=5


=== SMOTE ===
tp: 0.0198, fp: 0.0440, fn: 0.0982, tn: 0.8380
Accuracy: 0.8578, Precision: 0.3102, Recall: 0.1678, F1: 0.2178

=== BorderlineSMOTE ===
tp: 0.0206, fp: 0.0435, fn: 0.0974, tn: 0.8385
Accuracy: 0.8591, Precision: 0.3212, Recall: 0.1743, F1: 0.2260

=== ADASYN ===
tp: 0.0274, fp: 0.0683, fn: 0.0906, tn: 0.8137
Accuracy: 0.8411, Precision: 0.2864, Recall: 0.2324, F1: 0.2566

=== RandomUnderSampler ===
tp: 0.0719, fp: 0.3425, fn: 0.0461, tn: 0.5395
Accuracy: 0.6114, Precision: 0.1735, Recall: 0.6092, F1: 0.2700

=== SMOTETomek ===
tp: 0.0202, fp: 0.0451, fn: 0.0978, tn: 0.8369
Accuracy: 0.8571, Precision: 0.3096, Recall: 0.1715, F1: 0.2207

=== SMOTEENN ===
tp: 0.0736, fp: 0.3584, fn: 0.0443, tn: 0.5236
Accuracy: 0.5973, Precision: 0.1705, Recall: 0.6242, F1: 0.2678


=== Overall Comparison of Balancing Methods  k=5  n=10 ===
                        TP      FP      FN      TN  Accuracy  Precision  \
SMOTE               0.0198  0.0440  0.0982  0.8380    0.8578     0.3102   
BorderlineSMOTE     0.0206  0.0435  0.0974  0.8385    0.8591     0.3212   
ADASYN              0.0274  0.0683  0.0906  0.8137    0.8411     0.2864   
RandomUnderSampler  0.0719  0.3425  0.0461  0.5395    0.6114     0.1735   
SMOTETomek          0.0202  0.0451  0.0978  0.8369    0.8571     0.3096   
SMOTEENN            0.0736  0.3584  0.0443  0.5236    0.5973     0.1705   

                    Recall      F1  
SMOTE               0.1678  0.2178  
BorderlineSMOTE     0.1743  0.2260  
ADASYN              0.2324  0.2566  
RandomUnderSampler  0.6092  0.2700  
SMOTETomek          0.1715  0.2207  
SMOTEENN            0.6242  0.2678


=== Overall Comparison of Balancing Methods  k=5  (No PCA) ===
                        TP      FP      FN      TN  Accuracy  Precision  \
SMOTE               0.0641  0.2842  0.0539  0.5978    0.6619     0.1841   
BorderlineSMOTE     0.0529  0.2303  0.0651  0.6517    0.7045     0.1866   
ADASYN              0.0619  0.3014  0.0561  0.5806    0.6425     0.1704   
RandomUnderSampler  0.0761  0.3294  0.0419  0.5526    0.6287     0.1876   
SMOTETomek          0.0641  0.2842  0.0539  0.5978    0.6619     0.1841   
SMOTEENN            0.0853  0.4612  0.0327  0.4208    0.5060     0.1560   

                    Recall      F1  
SMOTE               0.5436  0.2751  
BorderlineSMOTE     0.4480  0.2635  
ADASYN              0.5248  0.2573  
RandomUnderSampler  0.6448  0.2907  
SMOTETomek          0.5436  0.2751  
SMOTEENN            0.7226  0.2566
```